In [6]:
!pip install --upgrade openai anthropic google-generativeai together pandas numpy


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [9]:
pip install --upgrade pip

  Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Not uninstalling pip at /usr/local/lib/python3.10/site-packages, outside environment /root/venv
    Can't uninstall 'pip'. No files were found to uninstall.

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
ABLATION STUDY - REVISED WITH FULL METHODOLOGICAL PROMPTS
==========================================================
This version uses prompts that match the GEN_DATA_COUNTRY format exactly,
ensuring that all prompts include the full survey methodology.

Key changes from original ablation:
1. Country_only now matches Stage 1 (includes full survey question)
2. All other ablations match Stage 8 format, removing only specified features
3. Maintains same ablation conditions and counterfactuals

FEATURES:
✓ Checkpoint/resume system
✓ Keep-alive thread for long runs
✓ Retry logic with exponential backoff
✓ All ablation conditions + counterfactuals
✓ Progress tracking and time estimates
✓ Forbidden string detection

USAGE:
1. First run: python ablation_study_revised_prompts.py
2. If interrupted: Just run again - will resume from checkpoint
"""

import os, re, time, json, threading
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# API imports
from openai import OpenAI
import anthropic
import google.generativeai as genai
from together import Together

print("="*80)
print("ABLATION STUDY - REVISED PROMPTS (FULL METHODOLOGY)")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# ================================================================
# Keep-alive thread
# ================================================================

def keep_alive():
    """Print periodic messages to keep session alive."""
    while True:
        if time.time() % 300 < 60:
            print(f"\n⏱️  [{datetime.now().strftime('%H:%M:%S')}] Session alive", flush=True)
        else:
            print(".", end="", flush=True)
        time.sleep(60)

keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("✓ Keep-alive thread started\n")

# ================================================================
# Configuration
# ================================================================

CHECKPOINT_FILE = "ablation_revised_checkpoint.json"
RAW_RESULTS_FILE = "ablation_revised_raw_results.csv"
DATA_FILE = "data_final.csv"

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
}

# System instruction
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# Condition order
condition_order = [
    'full',
    'no_econ',
    'no_religion',
    'no_demo',
    'no_climate',
    'no_own_willingness',
    'country_only',
    'cf_gdp_flip',
    'cf_name_mismatch',
    'cf_willingness_flip'
]

# ================================================================
# Helper Functions
# ================================================================

def as_num(x, nd=1):
    """Convert to number with specified decimal places."""
    if pd.isna(x): 
        return None
    try:
        return round(float(x), nd)
    except:
        return None

def as_pct(x):
    """Return percentage (0-100) as float. Accept 0–1 or 0–100."""
    if pd.isna(x): 
        return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:
        return v * 100.0
    return v

def fmt_pct(x):
    """Format as percentage string."""
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    """Format as number string."""
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

def extract_number_0_100(text):
    """Extract a number between 0-100 from text or JSON."""
    if not isinstance(text, str):
        return None
    
    # Try parsing as JSON first
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    # Fallback: regex extraction
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

def contains_forbidden_strings(text):
    """Check if response contains forbidden strings."""
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre",
        "et al", "et. al", "et.al",
        "doi", "http://", "https://",
        "paper", "study", "research",
        "published", "journal", "article"
    ]
    
    return any(term in text_lower for term in forbidden)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")
if not Path(DATA_FILE).exists():
    print(f"   ❌ Data file not found: {DATA_FILE}")
    exit(1)

df = pd.read_csv(DATA_FILE)
print(f"   ✓ Loaded {len(df)} countries")

# Identify column names
OWN_LESS_COL = None
for col_name in ["mean_own_willingness_less", "mean_own_willigness_less"]:
    if col_name in df.columns:
        OWN_LESS_COL = col_name
        break

TEMP_COL = None
for col_name in ["temp_mean", "temp_mean_2010_2019"]:
    if col_name in df.columns:
        TEMP_COL = col_name
        break

print(f"   ✓ Temperature column: {TEMP_COL}")
print(f"   ✓ Willingness_less column: {OWN_LESS_COL}")

# ================================================================
# REVISED PROMPT BUILDERS - MATCHING GEN_DATA_COUNTRY FORMAT
# ================================================================

def build_ablated_prompts_revised(row, all_countries_df):
    """
    Build all ablation versions matching GEN_DATA_COUNTRY format.
    Key principle: Use Stage 8 format, remove only the specific features being tested.
    """
    country = row["countrynew"]
    prompts = {}
    
    # ================================================================
    # BUILD REUSABLE COMPONENTS
    # ================================================================
    
    # Full survey methodology (used in ALL prompts)
    survey_intro = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, "
        f"respondents were asked: 'Would you be willing to contribute 1% of your household income every month to fight global warming? "
        f"This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. "
        f"Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). "
    )
    
    # Demographics - Full
    socio_full = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    
    # Demographics - No religion
    socio_no_religion = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"and {fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education."
    )
    
    # Economics - Full
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    # Economics - GDP only (for no_demo)
    gdp_only = f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}."
    
    # Religion only (for no_demo)
    religion_only = f"{fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    
    # Temperature
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    
    # Willingness
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    
    # ================================================================
    # TASK INSTRUCTIONS (with features listed)
    # ================================================================
    
    def make_task(features_list):
        """Generate task instruction with features listed."""
        features_text = ", ".join(features_list)
        return (
            f"Based on the country{', ' + features_text if features_text else ''}, "
            f"estimate what respondents in {country} on average thought about how many OTHER respondents in {country} "
            f"are willing to contribute at least 1% of their household income every month to fight global warming. "
            f"Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
            f"Respond with a single number between 0 and 100, with one decimal place."
        )
    
    # ================================================================
    # CONDITION 1: FULL (BASELINE - STAGE 8)
    # ================================================================
    
    prompts['full'] = (
        f"In {country}, {socio_full} {macro_economic} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 2: NO ECONOMIC INDICATORS
    # ================================================================
    
    prompts['no_econ'] = (
        f"In {country}, {socio_full} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 3: NO RELIGION
    # ================================================================
    
    prompts['no_religion'] = (
        f"In {country}, {socio_no_religion} {macro_economic} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 4: NO DEMOGRAPHICS (keep only GDP from econ)
    # ================================================================
    
    prompts['no_demo'] = (
        f"In {country}, {gdp_only} {religion_only} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['GDP per capita', 'religion importance', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 5: NO CLIMATE
    # ================================================================
    
    prompts['no_climate'] = (
        f"In {country}, {socio_full} {macro_economic} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 6: NO OWN WILLINGNESS (CRITICAL - removes "answer key")
    # ================================================================
    
    prompts['no_own_willingness'] = (
        f"In {country}, {socio_full} {macro_economic} {temperature} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'and temperature data'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 7: COUNTRY ONLY (MATCHES STAGE 1)
    # ================================================================
    
    prompts['country_only'] = (
        f"In {country}. "
        f"{survey_intro}"
        f"Based on the country, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} "
        f"are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        f"Respond with a single number between 0 and 100, with one decimal place."
    ).replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUAL 1: GDP FLIP
    # ================================================================
    
    gdp = row.get('gdp_capita_2021', 0)
    is_rich = gdp > 20000
    
    if is_rich:
        cf_gdp_str = "$2,000"
    else:
        cf_gdp_str = "$65,000"
    
    macro_economic_flipped = (
        f"GDP per capita (PPP, 2021) is {cf_gdp_str}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    prompts['cf_gdp_flip'] = (
        f"In {country}, {socio_full} {macro_economic_flipped} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUAL 2: NAME-DATA MISMATCH
    # ================================================================
    
    all_countries_sorted = sorted(all_countries_df['countrynew'].unique())
    country_idx = all_countries_sorted.index(country) if country in all_countries_sorted else 0
    
    if is_rich:
        poor_countries = all_countries_df[all_countries_df['gdp_capita_2021'] < 5000]['countrynew'].tolist()
        if poor_countries:
            cf_name = sorted(poor_countries)[country_idx % len(poor_countries)]
        else:
            cf_name = "Chad"
    else:
        rich_countries = all_countries_df[all_countries_df['gdp_capita_2021'] > 40000]['countrynew'].tolist()
        if rich_countries:
            cf_name = sorted(rich_countries)[country_idx % len(rich_countries)]
        else:
            cf_name = "Norway"
    
    # Replace country name in survey intro
    survey_intro_mismatch = survey_intro.replace(country, cf_name)
    task_mismatch = make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above']).replace(country, cf_name)
    
    prompts['cf_name_mismatch'] = (
        f"In {cf_name}, {socio_full} {macro_economic} {temperature} {willingness} "
        f"{survey_intro_mismatch}"
        f"{task_mismatch}"
    ).replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUAL 3: WILLINGNESS FLIP
    # ================================================================
    
    own_willingness = row.get('mean_own_willingness', 0)
    is_high_willingness = own_willingness > 50
    
    if is_high_willingness:
        cf_own_main = "15.0%"
        cf_own_less = "10.0%"
    else:
        cf_own_main = "75.0%"
        cf_own_less = "15.0%"
    
    willingness_flipped = (
        f"In this survey, {cf_own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {cf_own_less} would contribute a smaller amount."
    )
    
    prompts['cf_willingness_flip'] = (
        f"In {country}, {socio_full} {macro_economic} {temperature} {willingness_flipped} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    return prompts

# ================================================================
# API Calling Functions
# ================================================================

def call_gpt(prompt, model="gpt-4o-mini", max_retries=3):
    """Call OpenAI API with JSON mode."""
    if not OPENAI_API_KEY:
        return None
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ GPT attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=5):
    """Call Anthropic API."""
    if not CLAUDE_API_KEY:
        return None
    
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,
                system=SYSTEM_INSTRUCTION,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                tools=[],
            )
            content = response.content[0].text
            
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except anthropic.RateLimitError as e:
            wait_time = (2 ** attempt) * 2
            print(f"⚠️  Claude rate limit, waiting {wait_time}s...")
            if attempt < max_retries - 1:
                time.sleep(wait_time)
            else:
                print(f"❌ Claude: Rate limit exceeded")
                return None
        except Exception as e:
            print(f"❌ Claude attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-2.5-flash", max_retries=3):
    """Call Google Gemini API."""
    if not GEMINI_API_KEY:
        return None
    
    genai.configure(api_key=GEMINI_API_KEY)
    model_obj = genai.GenerativeModel(model)
    
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
    
    for attempt in range(max_retries):
        try:
            response = model_obj.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0,
                )
            )
            content = response.text
            
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Gemini attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", max_retries=3):
    """Call Llama via Together API."""
    if not LLAMA_API_KEY:
        return None
    
    client = Together(api_key=LLAMA_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Llama attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

# ================================================================
# Checkpoint Management
# ================================================================

def load_checkpoint():
    """Load checkpoint if it exists."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"completed": []}

def save_checkpoint(checkpoint):
    """Save checkpoint."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f)

def is_completed(checkpoint, country, condition, model):
    """Check if a specific combination was completed."""
    key = f"{country}|{condition}|{model}"
    return key in checkpoint["completed"]

def mark_completed(checkpoint, country, condition, model):
    """Mark a combination as completed."""
    key = f"{country}|{condition}|{model}"
    if key not in checkpoint["completed"]:
        checkpoint["completed"].append(key)

# ================================================================
# Main Ablation Study
# ================================================================

def run_ablation_study():
    """Run the complete ablation study."""
    
    print("\n2. Loading checkpoint...")
    checkpoint = load_checkpoint()
    print(f"   ✓ {len(checkpoint['completed'])} tasks already completed")
    
    # Initialize or load results
    if Path(RAW_RESULTS_FILE).exists():
        results_df = pd.read_csv(RAW_RESULTS_FILE)
        print(f"   ✓ Loaded existing results: {len(results_df)} rows")
    else:
        results_df = pd.DataFrame()
        print("   ✓ Starting fresh results file")
    
    # Calculate total tasks
    available_models = [m for m in MODELS.keys() if 
                       (m == "gpt" and OPENAI_API_KEY) or 
                       (m == "claude" and CLAUDE_API_KEY) or 
                       (m == "gemini" and GEMINI_API_KEY) or 
                       (m == "llama" and LLAMA_API_KEY)]
    
    total_tasks = len(df) * len(condition_order) * len(available_models)
    completed_tasks = len(checkpoint['completed'])
    
    print(f"\n3. Running ablation study...")
    print(f"   Total tasks: {total_tasks}")
    print(f"   Remaining: {total_tasks - completed_tasks}")
    print(f"   Progress: {100 * completed_tasks / total_tasks:.1f}%\n")
    
    results = []
    start_time = time.time()
    
    for i, (idx, row) in enumerate(df.iterrows()):
        country = row['countrynew']
        
        print(f"\n[{i+1}/{len(df)}] Processing: {country}")
        
        # Build all prompts for this country
        all_prompts = build_ablated_prompts_revised(row, df)
        
        # Test each condition
        for condition in condition_order:
            prompt = all_prompts[condition]
            
            # Test each model
            for model_name, model_id in MODELS.items():
                # Skip if already completed
                if is_completed(checkpoint, country, condition, model_name):
                    continue
                
                # Check if API key is available
                if model_name == "gpt" and not OPENAI_API_KEY:
                    continue
                if model_name == "claude" and not CLAUDE_API_KEY:
                    continue
                if model_name == "gemini" and not GEMINI_API_KEY:
                    continue
                if model_name == "llama" and not LLAMA_API_KEY:
                    continue
                
                print(f"  [{condition:20s}] [{model_name:7s}] ", end="", flush=True)
                
                # Call the appropriate API
                if model_name == "gpt":
                    prediction = call_gpt(prompt, model_id)
                elif model_name == "claude":
                    prediction = call_claude(prompt, model_id)
                elif model_name == "gemini":
                    prediction = call_gemini(prompt, model_id)
                elif model_name == "llama":
                    prediction = call_llama(prompt, model_id)
                else:
                    prediction = None
                
                if prediction is not None:
                    print(f"✓ {prediction:.1f}")
                else:
                    print(f"✗ Failed")
                
                # Save result
                results.append({
                    'country': country,
                    'condition': condition,
                    'model': model_name,
                    'prediction': prediction,
                    'actual_other_willingness': row.get('mean_other_willingness'),
                    'actual_own_willingness': row.get('mean_own_willingness'),
                })
                
                # Mark as completed
                mark_completed(checkpoint, country, condition, model_name)
                completed_tasks += 1
                
                # Save checkpoint every 10 tasks
                if completed_tasks % 10 == 0:
                    save_checkpoint(checkpoint)
                    
                    # Save results
                    if results:
                        new_results_df = pd.DataFrame(results)
                        if len(results_df) > 0:
                            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
                        else:
                            results_df = new_results_df
                        results_df.to_csv(RAW_RESULTS_FILE, index=False)
                        results = []
                    
                    # Time estimate
                    elapsed = time.time() - start_time
                    rate = completed_tasks / elapsed
                    remaining = total_tasks - completed_tasks
                    eta_seconds = remaining / rate if rate > 0 else 0
                    eta_minutes = eta_seconds / 60
                    
                    print(f"\n  💾 Checkpoint saved | Progress: {100 * completed_tasks / total_tasks:.1f}% | ETA: {eta_minutes:.1f} min\n")
    
    # Final save
    if results:
        new_results_df = pd.DataFrame(results)
        if len(results_df) > 0:
            results_df = pd.concat([results_df, new_results_df], ignore_index=True)
        else:
            results_df = new_results_df
        results_df.to_csv(RAW_RESULTS_FILE, index=False)
    
    save_checkpoint(checkpoint)
    
    print("\n" + "="*80)
    print("ABLATION STUDY COMPLETE!")
    print("="*80)
    print(f"Results saved to: {RAW_RESULTS_FILE}")
    print(f"Total predictions: {len(results_df)}")
    print(f"Time elapsed: {(time.time() - start_time) / 60:.1f} minutes")
    
    return results_df

# ================================================================
# Run the study
# ================================================================

if __name__ == "__main__":
    try:
        results = run_ablation_study()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
        print("Progress has been saved. Run again to resume.")
    except Exception as e:
        print(f"\n\n❌ Error: {e}")
        print("Progress has been saved. Run again to resume.")
        raise

╭───────────────────────────────────────────── 🚀 New SDK Available ──────────────────────────────────────────────╮
│ Together Python SDK 2.0 is now available!                                                                       │
│                                                                                                                 │
│ Install the beta:                                                                                               │
│ pip install --pre together  or  uv add together --prerelease allow                                              │
│                                                                                                                 │
│ New SDK: ]8;id=37958;https://github.com/togethercomputer/together-py\https://github.com/togethercomputer/together-py]8;;\                                                        │
│ Migration guide: ]8;id=217057;https://docs.together.ai/docs/pythonv2-migration-guide\https://docs.together.ai/docs/pythonv2-migration-guide]8;;\                                         │
│                                                                                                                 │
│ This package will be maintained until January 2026.                                                             │
│ Set TOGETHER_NO_BANNER=1 to hide this message.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  [no_econ             ] [gemini ] ✓ 45.5
  [no_econ             ] [llama  ] ✓ 43.8
  [no_religion         ] [gpt    ] ✓ 50.0
  [no_religion         ] [claude ] ✓ 42.7
.
  💾 Checkpoint saved | Progress: 85.8% | ETA: 74.3 min

  [no_religion         ] [gemini ] ✓ 48.5
  [no_religion         ] [llama  ] ✓ 34.4
  [no_demo             ] [gpt    ] ✓ 50.0
  [no_demo             ] [claude ] ✓ 42.3
  [no_demo             ] [gemini ] ✓ 43.5
  [no_demo             ] [llama  ] ✓ 43.8
  [no_climate          ] [gpt    ] ✓ 50.0
  [no_climate          ] [claude ] ✓ 42.7
  [no_climate          ] [gemini ] .✓ 48.9
  [no_climate          ] [llama  ] ✓ 43.8

  💾 Checkpoint saved | Progress: 86.0% | ETA: 73.2 min

  [no_own_willingness  ] [gpt    ] ✓ 45.0
  [no_own_willingness  ] [claude ] ✓ 42.7
  [no_own_willingness  ] [gemini ] ✓ 22.5
  [no_own_willingness  ] [llama  ] ✓ 34.5
  [country_only        ] [gpt    ] ✓ 45.0
  [country_only        ] [claude ] ✓ 35.7
  [country_only        ] [gemini ] ✓ 32.5
  

In [ ]:
"""
FILL MISSING PREDICTIONS - REVISED NARRATIVE ABLATION
======================================================
Identifies gaps in the revised narrative ablation results and re-runs
only the missing API calls.

This script uses the EXACT SAME prompts as ablation_study_revised_prompts.py
to ensure consistency.

USAGE:
python fill_missing_revised.py
"""

import os, re, time, json
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path

# API imports
from openai import OpenAI
import anthropic
import google.generativeai as genai
from together import Together

print("="*80)
print("FILLING MISSING PREDICTIONS - REVISED NARRATIVE ABLATION")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# ================================================================
# Configuration
# ================================================================

RAW_RESULTS_FILE = "ablation_revised_raw_results.csv"
OUTPUT_FILE = "ablation_revised_raw_results_complete.csv"
DATA_FILE = "data_final.csv"

# API Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"
}

# System instruction
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# Condition order
condition_order = [
    'full',
    'no_econ',
    'no_religion',
    'no_demo',
    'no_climate',
    'no_own_willingness',
    'country_only',
    'cf_gdp_flip',
    'cf_name_mismatch',
    'cf_willingness_flip'
]

# ================================================================
# Helper Functions
# ================================================================

def as_num(x, nd=1):
    if pd.isna(x): 
        return None
    try:
        return round(float(x), nd)
    except:
        return None

def as_pct(x):
    if pd.isna(x): 
        return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:
        return v * 100.0
    return v

def fmt_pct(x):
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

def extract_number_0_100(text):
    if not isinstance(text, str):
        return None
    
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

def contains_forbidden_strings(text):
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre", "et al", "et. al", "et.al",
        "doi", "http://", "https://",
        "paper", "study", "research",
        "published", "journal", "article"
    ]
    
    return any(term in text_lower for term in forbidden)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")

if not Path(DATA_FILE).exists():
    print(f"   ❌ Data file not found: {DATA_FILE}")
    exit(1)

df = pd.read_csv(DATA_FILE)
print(f"   ✓ Loaded {len(df)} countries")

# Column identification
OWN_LESS_COL = None
for col_name in ["mean_own_willingness_less", "mean_own_willigness_less"]:
    if col_name in df.columns:
        OWN_LESS_COL = col_name
        break

TEMP_COL = None
for col_name in ["temp_mean", "temp_mean_2010_2019"]:
    if col_name in df.columns:
        TEMP_COL = col_name
        break

print(f"   ✓ Temperature column: {TEMP_COL}")
print(f"   ✓ Willingness_less column: {OWN_LESS_COL}")

if not Path(RAW_RESULTS_FILE).exists():
    print(f"   ❌ Results file not found: {RAW_RESULTS_FILE}")
    print("   Run the main ablation study first!")
    exit(1)

results_df = pd.read_csv(RAW_RESULTS_FILE)
print(f"   ✓ Loaded existing results: {len(results_df)} rows")

# ================================================================
# REVISED PROMPT BUILDERS - COPIED FROM MAIN SCRIPT
# ================================================================

def build_ablated_prompts_revised(row, all_countries_df):
    """
    Build all ablation versions matching GEN_DATA_COUNTRY format.
    EXACT COPY from ablation_study_revised_prompts.py
    """
    country = row["countrynew"]
    prompts = {}
    
    # ================================================================
    # BUILD REUSABLE COMPONENTS
    # ================================================================
    
    # Full survey methodology (used in ALL prompts)
    survey_intro = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, "
        f"respondents were asked: 'Would you be willing to contribute 1% of your household income every month to fight global warming? "
        f"This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. "
        f"Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). "
    )
    
    # Demographics - Full
    socio_full = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    
    # Demographics - No religion
    socio_no_religion = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"and {fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education."
    )
    
    # Economics - Full
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    # Economics - GDP only (for no_demo)
    gdp_only = f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}."
    
    # Religion only (for no_demo)
    religion_only = f"{fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    
    # Temperature
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    
    # Willingness
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    
    # ================================================================
    # TASK INSTRUCTIONS (with features listed)
    # ================================================================
    
    def make_task(features_list):
        """Generate task instruction with features listed."""
        features_text = ", ".join(features_list)
        return (
            f"Based on the country{', ' + features_text if features_text else ''}, "
            f"estimate what respondents in {country} on average thought about how many OTHER respondents in {country} "
            f"are willing to contribute at least 1% of their household income every month to fight global warming. "
            f"Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
            f"Respond with a single number between 0 and 100, with one decimal place."
        )
    
    # ================================================================
    # CONDITION 1: FULL (BASELINE - STAGE 8)
    # ================================================================
    
    prompts['full'] = (
        f"In {country}, {socio_full} {macro_economic} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 2: NO ECONOMIC INDICATORS
    # ================================================================
    
    prompts['no_econ'] = (
        f"In {country}, {socio_full} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 3: NO RELIGION
    # ================================================================
    
    prompts['no_religion'] = (
        f"In {country}, {socio_no_religion} {macro_economic} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 4: NO DEMOGRAPHICS (keep only GDP from econ)
    # ================================================================
    
    prompts['no_demo'] = (
        f"In {country}, {gdp_only} {religion_only} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['GDP per capita', 'religion importance', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 5: NO CLIMATE
    # ================================================================
    
    prompts['no_climate'] = (
        f"In {country}, {socio_full} {macro_economic} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 6: NO OWN WILLINGNESS (CRITICAL - removes "answer key")
    # ================================================================
    
    prompts['no_own_willingness'] = (
        f"In {country}, {socio_full} {macro_economic} {temperature} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'and temperature data'])}"
    ).replace("..", ".")
    
    # ================================================================
    # CONDITION 7: COUNTRY ONLY (MATCHES STAGE 1)
    # ================================================================
    
    prompts['country_only'] = (
        f"In {country}. "
        f"{survey_intro}"
        f"Based on the country, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} "
        f"are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        f"Respond with a single number between 0 and 100, with one decimal place."
    ).replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUAL 1: GDP FLIP
    # ================================================================
    
    gdp = row.get('gdp_capita_2021', 0)
    is_rich = gdp > 20000
    
    if is_rich:
        cf_gdp_str = "$2,000"
    else:
        cf_gdp_str = "$65,000"
    
    macro_economic_flipped = (
        f"GDP per capita (PPP, 2021) is {cf_gdp_str}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    
    prompts['cf_gdp_flip'] = (
        f"In {country}, {socio_full} {macro_economic_flipped} {temperature} {willingness} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUAL 2: NAME-DATA MISMATCH
    # ================================================================
    
    all_countries_sorted = sorted(all_countries_df['countrynew'].unique())
    country_idx = all_countries_sorted.index(country) if country in all_countries_sorted else 0
    
    if is_rich:
        poor_countries = all_countries_df[all_countries_df['gdp_capita_2021'] < 5000]['countrynew'].tolist()
        if poor_countries:
            cf_name = sorted(poor_countries)[country_idx % len(poor_countries)]
        else:
            cf_name = "Chad"
    else:
        rich_countries = all_countries_df[all_countries_df['gdp_capita_2021'] > 40000]['countrynew'].tolist()
        if rich_countries:
            cf_name = sorted(rich_countries)[country_idx % len(rich_countries)]
        else:
            cf_name = "Norway"
    
    # Replace country name in survey intro
    survey_intro_mismatch = survey_intro.replace(country, cf_name)
    task_mismatch = make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above']).replace(country, cf_name)
    
    prompts['cf_name_mismatch'] = (
        f"In {cf_name}, {socio_full} {macro_economic} {temperature} {willingness} "
        f"{survey_intro_mismatch}"
        f"{task_mismatch}"
    ).replace("..", ".")
    
    # ================================================================
    # COUNTERFACTUAL 3: WILLINGNESS FLIP
    # ================================================================
    
    own_willingness = row.get('mean_own_willingness', 0)
    is_high_willingness = own_willingness > 50
    
    if is_high_willingness:
        cf_own_main = "15.0%"
        cf_own_less = "10.0%"
    else:
        cf_own_main = "75.0%"
        cf_own_less = "15.0%"
    
    willingness_flipped = (
        f"In this survey, {cf_own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {cf_own_less} would contribute a smaller amount."
    )
    
    prompts['cf_willingness_flip'] = (
        f"In {country}, {socio_full} {macro_economic} {temperature} {willingness_flipped} "
        f"{survey_intro}"
        f"{make_task(['socio-demographic', 'macro-economic indicators', 'temperature data', 'and the actual willingness data shown above'])}"
    ).replace("..", ".")
    
    return prompts

# ================================================================
# API Functions
# ================================================================

def call_gpt(prompt, model="gpt-4o-mini", max_retries=3):
    if not OPENAI_API_KEY:
        return None
    client = OpenAI(api_key=OPENAI_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=5):
    if not CLAUDE_API_KEY:
        return None
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,
                system=SYSTEM_INSTRUCTION,
                messages=[{"role": "user", "content": prompt}],
                tools=[],
            )
            content = response.content[0].text
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except anthropic.RateLimitError:
            wait_time = (2 ** attempt) * 2
            if attempt < max_retries - 1:
                time.sleep(wait_time)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-2.5-flash", max_retries=3):
    if not GEMINI_API_KEY:
        return None
    genai.configure(api_key=GEMINI_API_KEY)
    model_obj = genai.GenerativeModel(model)
    full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
    for attempt in range(max_retries):
        try:
            response = model_obj.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(temperature=0)
            )
            content = response.text
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", max_retries=3):
    if not LLAMA_API_KEY:
        return None
    client = Together(api_key=LLAMA_API_KEY)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            if contains_forbidden_strings(content):
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return None
            return extract_number_0_100(content)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

# ================================================================
# Identify Missing
# ================================================================

print("\n2. Identifying missing predictions...")

expected = []
for _, row in df.iterrows():
    for condition in condition_order:
        for model_name in MODELS.keys():
            expected.append({
                'country': row['countrynew'],
                'condition': condition,
                'model': model_name
            })

expected_df = pd.DataFrame(expected)
print(f"   ✓ Expected: {len(expected_df)} predictions")

merged = expected_df.merge(
    results_df,
    on=['country', 'condition', 'model'],
    how='left',
    indicator=True
)

missing = merged[merged['_merge'] == 'left_only'][['country', 'condition', 'model']]
print(f"   ✓ Missing: {len(missing)} predictions")

if len(missing) == 0:
    print("\n✓ No missing predictions! Dataset is complete.")
    print(f"Saving complete dataset to: {OUTPUT_FILE}")
    results_df.to_csv(OUTPUT_FILE, index=False)
    exit(0)

print(f"\n3. Filling {len(missing)} missing predictions...")

# Fill missing
new_results = []

for i, (idx, miss) in enumerate(missing.iterrows()):
    country = miss['country']
    condition = miss['condition']
    model_name = miss['model']
    
    country_row = df[df['countrynew'] == country].iloc[0]
    
    # Build ALL prompts
    all_prompts = build_ablated_prompts_revised(country_row, df)
    prompt = all_prompts.get(condition, '')
    
    if not prompt:
        print(f"[{i+1}/{len(missing)}] {country:20s} | {condition:20s} | {model_name:7s} ✗ No prompt")
        continue
    
    print(f"[{i+1}/{len(missing)}] {country:20s} | {condition:20s} | {model_name:7s} ", end="", flush=True)
    
    if model_name == "gpt":
        prediction = call_gpt(prompt, MODELS[model_name])
    elif model_name == "claude":
        prediction = call_claude(prompt, MODELS[model_name])
    elif model_name == "gemini":
        prediction = call_gemini(prompt, MODELS[model_name])
    elif model_name == "llama":
        prediction = call_llama(prompt, MODELS[model_name])
    else:
        prediction = None
    
    if prediction is not None:
        print(f"✓ {prediction:.1f}")
    else:
        print(f"✗ Failed")
    
    new_results.append({
        'country': country,
        'condition': condition,
        'model': model_name,
        'prediction': prediction,
        'actual_other_willingness': country_row.get('mean_other_willingness'),
        'actual_own_willingness': country_row.get('mean_own_willingness'),
    })
    
    if (i + 1) % 10 == 0:
        temp_df = pd.DataFrame(new_results)
        temp_updated = pd.concat([results_df, temp_df], ignore_index=True)
        temp_updated.to_csv(OUTPUT_FILE, index=False)
        print(f"  💾 Progress saved ({len(temp_updated)} total rows)\n")

# Final save
new_df = pd.DataFrame(new_results)
updated_df = pd.concat([results_df, new_df], ignore_index=True)
updated_df.to_csv(OUTPUT_FILE, index=False)

print("\n" + "="*80)
print("FILLING COMPLETE!")
print("="*80)
print(f"Complete dataset saved to: {OUTPUT_FILE}")
print(f"Total predictions: {len(updated_df)}")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>